# Figure 3 — cascading healthcare-service failures

Draft extracted from `moz_paper_figure.ipynb`, cell index 10. The four compact
GeoPackages are regenerated from the current non-ablation public-grid,
downstream water/communication, and direct-wind results by
`prepare_failure_figure_inputs.py`.

The `-1` sentinel is displayed as unavailable/uncertain. Confirm the final
annualization factor and the precise eligible-facility denominators before
publication.


In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("INFRACASCADE_PROJECT_DIR", Path.cwd())).resolve()
if PROJECT_ROOT.name == "moz_notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = Path(os.environ.get("MOZ_FIGURE_DATA_ROOT", PROJECT_ROOT / "figure_data"))
FIGURE_DIR = Path(os.environ.get(
    "MOZ_MANUSCRIPT_FIGURE_DIR",
    PROJECT_ROOT / "outputs" / "manuscript_figures",
))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)
print("Figures:", FIGURE_DIR)


In [ ]:
from capacity_ablation_figure import load_boundary

WORLD_PATH = Path(os.environ.get(
    "INFRACASCADE_BOUNDARY_DATA", DATA_ROOT / "World_Countries"
))
ELECTRICITY_PATH = DATA_ROOT / "df_final_public_25.gpkg"
WATER_PATH = DATA_ROOT / "water_hospital_mean.gpkg"
COMMUNICATION_PATH = DATA_ROOT / "communication_hospital_mean.gpkg"
DIRECT_PATH = DATA_ROOT / "direct_hospital_mean.gpkg"

mozambique = load_boundary(WORLD_PATH)
layers = [
    (gpd.read_file(ELECTRICITY_PATH), "mean", "(a) Electricity"),
    (gpd.read_file(WATER_PATH), "water_mean", "(b) Water"),
    (gpd.read_file(COMMUNICATION_PATH), "comm_mean", "(c) Communication"),
    (gpd.read_file(DIRECT_PATH), "direct", "(d) Direct"),
]

ANNUALIZATION_FACTOR = 2703 / 10_000


In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

cmap = "BuGn"
norm = Normalize(0, 0.05)
fig, axes = plt.subplots(1, 4, figsize=(12, 4.4), sharex=True, sharey=True)
fig.subplots_adjust(left=0.02, right=0.88, bottom=0.12, top=0.95, wspace=0.02)

for axis, (layer, value_column, title) in zip(axes, layers):
    mozambique.plot(ax=axis, color="gainsboro", edgecolor="black", linewidth=0.8)
    layer = layer.copy()
    unavailable = layer[layer[value_column].eq(-1)]
    valid = layer[layer[value_column].ge(0)].copy()
    valid["annual_probability"] = valid[value_column] * ANNUALIZATION_FACTOR

    if not unavailable.empty:
        unavailable.plot(
            ax=axis, color="silver", edgecolor="black", markersize=14, linewidth=0.25
        )
    valid.plot(
        ax=axis,
        column="annual_probability",
        cmap=cmap,
        norm=norm,
        markersize=15,
        linewidth=0,
    )
    axis.set_title(title, loc="left", fontsize=10)
    axis.set(xlim=(30, 41), ylim=(-27, -10))
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_aspect("equal")

cbar_axis = fig.add_axes([0.90, 0.18, 0.015, 0.68])
colorbar = fig.colorbar(
    cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_axis
)
colorbar.set_label("Annualized service disruption index (yr$^{-1}$)")
colorbar.set_ticks(np.arange(0, 0.051, 0.01), labels=[f"{x:.2f}" for x in np.arange(0, 0.051, 0.01)])
fig.legend(
    handles=[Line2D([], [], marker="o", linestyle="none", color="silver",
                    markeredgecolor="black", label="Data missing")],
    loc="lower center",
    frameon=False,
)
for suffix in ("pdf", "png"):
    fig.savefig(FIGURE_DIR / f"figure_03.{suffix}", dpi=300, bbox_inches="tight")
plt.show()
